# Task 2: Exploratory Data Analysis — Ethiopia Financial Inclusion

**Project:** Forecasting Financial Inclusion in Ethiopia  
**Notebook:** 02 — Full Exploratory Data Analysis  
**Author:** Abigiya Haile  
**Date:** 2025-07-19  

---

## Contents

1. Dataset Overview — temporal coverage, record summaries, data quality
2. Access Analysis — account ownership trajectory (2011–2024), growth rates, 2021–2024 slowdown
3. Usage Analysis — mobile money penetration, P2P transactions, registered vs. active gap
4. Event Timeline — catalogued events overlaid on indicator trend charts
5. Key Insights — 6 documented insights with supporting evidence
6. Data Limitations — gaps, caveats, and quality assessment

In [ ]:
# ─── Imports ─────────────────────────────────────────────────────────────────
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import (
    load_unified_data, get_observations, get_events, get_targets,
    get_by_pillar, join_events_to_impact_links,
)
from src.visualization import (
    plot_record_type_distribution, plot_pillar_distribution,
    plot_confidence_distribution, plot_source_type_distribution,
    plot_temporal_coverage, plot_account_ownership_trajectory,
    plot_gender_gap, plot_mobile_money_penetration, plot_event_timeline,
)

PILLAR_COLORS = {
    'ACCESS': '#2196F3', 'USAGE': '#4CAF50', 'GENDER': '#E91E63',
    'AFFORDABILITY': '#FF9800', 'QUALITY': '#9C27B0',
    'TRUST': '#00BCD4', 'DEPTH': '#795548',
}

print('Libraries loaded ✓')

In [ ]:
# ─── Load enriched dataset ───────────────────────────────────────────────────
ENRICHED_CSV = ROOT / 'data' / 'processed' / 'ethiopia_fi_enriched.csv'

if not ENRICHED_CSV.exists():
    print('[!] Enriched CSV not found — running generator...')
    import subprocess
    subprocess.run(['python', str(ROOT / 'scripts' / 'generate_enriched_csv.py')], check=True)

df = load_unified_data(ENRICHED_CSV)
obs_all = get_observations(df)
events   = get_events(df)
targets  = get_targets(df)

print(f'[✓] Loaded {len(df)} records  |  observations={len(obs_all)}  events={len(events)}  targets={len(targets)}')

---
## 1. Dataset Overview

In [ ]:
# ─── Record type & pillar distributions ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

rt = df['record_type'].value_counts()
axes[0].bar(rt.index, rt.values, color=['#2196F3','#4CAF50','#FF9800','#9C27B0'][:len(rt)])
for rect, v in zip(axes[0].patches, rt.values):
    axes[0].text(rect.get_x()+rect.get_width()/2, rect.get_height()+0.2, str(v),
                 ha='center', fontsize=12, fontweight='bold')
axes[0].set_title('Records by Type', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].spines[['top','right']].set_visible(False)

pillar_df = df[df['pillar'].notna() & (df['pillar'].astype(str).isin(list(PILLAR_COLORS.keys())))]
p = pillar_df['pillar'].value_counts()
clrs = [PILLAR_COLORS.get(x,'#607D8B') for x in p.index]
axes[1].barh(p.index, p.values, color=clrs)
for rect, v in zip(axes[1].patches, p.values):
    axes[1].text(rect.get_width()+0.1, rect.get_y()+rect.get_height()/2, str(v),
                 va='center', fontsize=11, fontweight='bold')
axes[1].set_title('Records by Pillar', fontsize=13, fontweight='bold')
axes[1].invert_yaxis()
axes[1].spines[['top','right']].set_visible(False)

plt.suptitle('Ethiopia Financial Inclusion — Dataset Overview', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Temporal coverage by indicator ──────────────────────────────────────────
fig = plot_temporal_coverage(df)
plt.show()

In [ ]:
# ─── Source type & confidence (data quality) ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

st = df['source_type'].value_counts()
axes[0].bar(st.index, st.values, color='#5C6BC0')
for rect, v in zip(axes[0].patches, st.values):
    axes[0].text(rect.get_x()+rect.get_width()/2, rect.get_height()+0.1, str(v),
                 ha='center', fontsize=10, fontweight='bold')
axes[0].set_title('Records by Source Type', fontsize=12, fontweight='bold')
axes[0].tick_params(axis='x', rotation=30)
axes[0].spines[['top','right']].set_visible(False)

conf = df['confidence'].dropna().value_counts()
conf_colors = {'high':'#4CAF50','medium':'#FF9800','low':'#F44336','estimated':'#9E9E9E'}
clrs_c = [conf_colors.get(c,'#607D8B') for c in conf.index]
wedges, texts, autos = axes[1].pie(
    conf.values, labels=conf.index, autopct='%1.0f%%', colors=clrs_c,
    startangle=90, textprops={'fontsize':11})
for a in autos: a.set_fontweight('bold')
axes[1].set_title('Data Quality: Confidence Levels', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'\nData Quality Assessment:')
print(f'  High confidence records : {conf.get("high",0)} ({conf.get("high",0)/len(df)*100:.0f}%)')
print(f'  Medium confidence records: {conf.get("medium",0)} ({conf.get("medium",0)/len(df)*100:.0f}%)')
print(f'  Primary sources (survey/regulator/operator): {df["source_type"].isin(["survey","regulator","operator"]).sum()} records')

---
## 2. Access Analysis

### 2.1 Account Ownership Trajectory (2011–2024)

In [ ]:
# ─── Filter account ownership data ────────────────────────────────────────────
acc = df[
    (df['record_type'].isin(['observation','target'])) &
    (df['indicator_code'] == 'ACC_OWNERSHIP')
].copy()
acc['observation_date'] = pd.to_datetime(acc['observation_date'], errors='coerce')
acc['value_numeric'] = pd.to_numeric(acc['value_numeric'], errors='coerce')

acc_all = acc[acc['gender'].fillna('all') == 'all'].sort_values('observation_date')
obs_only = acc_all[acc_all['record_type'] == 'observation']
targets_only = acc_all[acc_all['record_type'] == 'target']

# Growth rate calculations
print('=== Account Ownership Growth Rates ===')
vals = obs_only[['observation_date','value_numeric']].dropna()
for i in range(1, len(vals)):
    y1, v1 = vals.iloc[i-1]['observation_date'].year, vals.iloc[i-1]['value_numeric']
    y2, v2 = vals.iloc[i]['observation_date'].year,   vals.iloc[i]['value_numeric']
    pp = v2 - v1
    yrs = y2 - y1
    print(f'  {y1}→{y2}: +{pp:.0f} pp over {yrs} yrs  ({pp/yrs:.1f} pp/yr)')

In [ ]:
# ─── Account ownership trajectory plot ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))

ax.plot(obs_only['observation_date'], obs_only['value_numeric'],
        'o-', color='#2196F3', linewidth=2.5, markersize=9, zorder=4, label='Account Ownership (%)')

for _, row in obs_only.iterrows():
    ax.annotate(f"{row['value_numeric']:.0f}%",
                xy=(row['observation_date'], row['value_numeric']),
                xytext=(0, 12), textcoords='offset points',
                ha='center', fontsize=11, fontweight='bold', color='#1565C0')

# Target
if not targets_only.empty:
    ax.scatter(targets_only['observation_date'], targets_only['value_numeric'],
               s=160, marker='*', color='#FF9800', zorder=5, label='NFIS-II Target (70%, 2025)')
    for _, row in targets_only.iterrows():
        ax.annotate(f"{row['value_numeric']:.0f}% target",
                    xy=(row['observation_date'], row['value_numeric']),
                    xytext=(8, -15), textcoords='offset points',
                    ha='left', fontsize=10, color='#E65100')

# Highlight 2021–2024 slowdown
ax.axvspan(pd.Timestamp('2021-01-01'), pd.Timestamp('2024-12-31'),
           alpha=0.08, color='#F44336', label='Slowdown period (2021–2024)')
ax.text(pd.Timestamp('2022-06-01'), 52, '+3 pp over 3 yrs\n(despite Telebirr + M-Pesa)',
        fontsize=9.5, color='#B71C1C', ha='center', style='italic')

ax.set_ylim(0, 90)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.set_title('Ethiopia: Account Ownership Trajectory (2011–2024) with NFIS-II Target',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Adults with an Account (%)')
ax.legend(fontsize=10)
ax.set_facecolor('#F8FBFF')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Investigation of the 2021–2024 slowdown ──────────────────────────────────
print('=== Investigation: 2021–2024 Slowdown (+3 pp despite mobile money expansion) ===')
print()
print('1. Account Ownership (Findex, total accounts):')
print('   2021: 46%  →  2024: 49%  (+3 pp in 3 years, slowest period on record)')
print()
print('2. But Mobile Money Metrics tell a different story:')

mm = df[(df['record_type']=='observation') & (df['indicator_code']=='ACC_MM_ACCOUNT') &
        (df['gender'].fillna('all')=='all')].copy()
mm['value_numeric'] = pd.to_numeric(mm['value_numeric'], errors='coerce')
mm = mm.dropna(subset=['value_numeric']).sort_values('observation_date')
for _, r in mm.iterrows():
    print(f'   MM Account Rate {str(r["observation_date"])[:10]}: {r["value_numeric"]:.1f}%')

print()
print('3. Possible explanations for the divergence:')
print('   a) Findex measures ACTIVE accounts; many Telebirr registrations may be dormant')
print('   b) Active user rate ~30.5% of registered = large inactive base')
print('   c) Telebirr reached 40M+ subscribers but Findex only counts those who USED the account')
print('   d) Rural network gaps limit active usage despite registration')
print('   e) High agent density (750K+) concentrated in urban areas')

### 2.2 Gender Gap in Account Ownership

In [ ]:
# ─── Gender gap chart ────────────────────────────────────────────────────────
fig = plot_gender_gap(df)
plt.show()

gender_gap = df[(df['record_type']=='observation') & (df['indicator_code']=='GEN_GAP_ACC')].copy()
gender_gap['value_numeric'] = pd.to_numeric(gender_gap['value_numeric'], errors='coerce')
print('Gender gap (percentage points, male − female):')
for _, r in gender_gap.iterrows():
    print(f'  {str(r["observation_date"])[:10]}: {r["value_numeric"]:.0f} pp gap')

---
## 3. Usage Analysis

In [ ]:
# ─── Mobile money account penetration ────────────────────────────────────────
fig = plot_mobile_money_penetration(df)
plt.show()

In [ ]:
# ─── P2P transaction volume trend ────────────────────────────────────────────
p2p = df[
    (df['record_type']=='observation') &
    (df['indicator_code']=='USG_P2P_COUNT')
].copy()
p2p['observation_date'] = pd.to_datetime(p2p['observation_date'], errors='coerce')
p2p['value_numeric'] = pd.to_numeric(p2p['value_numeric'], errors='coerce')
p2p = p2p.dropna(subset=['value_numeric']).sort_values('observation_date')

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(p2p['fiscal_year'].fillna(p2p['observation_date'].dt.year.astype(str)),
       p2p['value_numeric'] / 1e6, color='#4CAF50', alpha=0.85)

for i, (_, row) in enumerate(p2p.iterrows()):
    ax.text(i, row['value_numeric']/1e6 + 2, f"{row['value_numeric']/1e6:.0f}M",
            ha='center', fontsize=11, fontweight='bold', color='#1B5E20')

if len(p2p) >= 2:
    growth = (p2p['value_numeric'].iloc[-1] / p2p['value_numeric'].iloc[-2] - 1) * 100
    ax.text(0.98, 0.92, f'+{growth:.0f}% YoY growth', transform=ax.transAxes,
            ha='right', fontsize=12, color='#388E3C',
            bbox=dict(boxstyle='round,pad=0.3', facecolor='#E8F5E9', edgecolor='#4CAF50'))

ax.set_title('P2P Transaction Count by Fiscal Year', fontsize=14, fontweight='bold')
ax.set_ylabel('Transactions (Millions)')
ax.set_xlabel('Fiscal Year')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Registered vs. Active Gap analysis ──────────────────────────────────────
active_rec = df[df['indicator_code']=='USG_MM_ACTIVE_RATE'].copy()
active_rec['value_numeric'] = pd.to_numeric(active_rec['value_numeric'], errors='coerce')

fig, ax = plt.subplots(figsize=(9, 5))

categories = ['Registered MM Accounts\n(% adults, 2024)', 'Active MM Accounts\n(% of registered)']
# MM account rate = 9.45% of adults; active rate = 30.5% of registered
registered_pct = 9.45
active_pct = active_rec['value_numeric'].iloc[0] if not active_rec.empty else 30.5
active_of_adults = registered_pct * active_pct / 100

bars = ax.bar(
    ['Registered\n(% adults)', 'Active\n(% adults)', 'Inactive\n(% adults)'],
    [registered_pct, active_of_adults, registered_pct - active_of_adults],
    color=['#4CAF50', '#2196F3', '#EF9A9A'],
    alpha=0.85
)
ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=12, fontweight='bold')

ax.set_title(f'Registered vs. Active Mobile Money Accounts (2024)\nActive rate: {active_pct:.0f}% of registered',
             fontsize=13, fontweight='bold')
ax.set_ylabel('% of adult population')
ax.set_ylim(0, 14)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Key finding: Only {active_of_adults:.1f}% of adults actively use mobile money')
print(f'despite {registered_pct:.1f}% having registered accounts — a {registered_pct-active_of_adults:.1f} pp dormancy gap.')

---
## 4. Event Timeline

In [ ]:
# ─── Event timeline overlaid on ACC_OWNERSHIP ────────────────────────────────
cat_colors = {
    'product_launch':'#E53935','market_entry':'#8E24AA','market_exit':'#757575',
    'policy':'#1E88E5','regulation':'#00ACC1','infrastructure':'#43A047',
    'partnership':'#FB8C00','milestone':'#F9A825','economic':'#6D4C41','pricing':'#F48FB1',
}

ev = events.copy()
ev['observation_date'] = pd.to_datetime(ev['observation_date'], errors='coerce')
ev = ev.dropna(subset=['observation_date']).sort_values('observation_date')

obs_acc = obs_all[
    (obs_all['indicator_code']=='ACC_OWNERSHIP') &
    (obs_all['gender'].fillna('all')=='all')
].copy()
obs_acc['observation_date'] = pd.to_datetime(obs_acc['observation_date'], errors='coerce')
obs_acc['value_numeric'] = pd.to_numeric(obs_acc['value_numeric'], errors='coerce')
obs_acc = obs_acc.dropna(subset=['observation_date','value_numeric']).sort_values('observation_date')

fig, ax = plt.subplots(figsize=(15, 7))

# Trend line
ax.plot(obs_acc['observation_date'], obs_acc['value_numeric'],
        'o-', color='#2196F3', linewidth=2.5, markersize=10, zorder=4, label='Account Ownership (%)')
for _, row in obs_acc.iterrows():
    ax.annotate(f"{row['value_numeric']:.0f}%",
                xy=(row['observation_date'], row['value_numeric']),
                xytext=(0, 13), textcoords='offset points',
                ha='center', fontsize=10, fontweight='bold', color='#1565C0')

# Event vertical lines
y_max = 80
for i, (_, ev_row) in enumerate(ev.iterrows()):
    clr = cat_colors.get(str(ev_row.get('category','')), '#78909C')
    ax.axvline(ev_row['observation_date'], color=clr, linewidth=1.5, alpha=0.65, linestyle='--', zorder=2)
    y_label = y_max - 8 - (i % 6) * 9
    short = str(ev_row.get('indicator','')).replace('Ethiopia','ET').replace('Digital','Dig.')
    ax.text(ev_row['observation_date'], y_label, short[:24],
            rotation=30, ha='left', va='top', fontsize=7.5, color=clr, fontweight='bold')

# Target
tgt = targets[targets['indicator_code']=='ACC_OWNERSHIP'].copy()
tgt['observation_date'] = pd.to_datetime(tgt['observation_date'], errors='coerce')
tgt['value_numeric'] = pd.to_numeric(tgt['value_numeric'], errors='coerce')
if not tgt.empty:
    ax.scatter(tgt['observation_date'], tgt['value_numeric'],
               s=180, marker='*', color='#FF9800', zorder=5, label='NFIS-II Target')

ax.set_ylim(0, y_max + 5)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(bymonth=[1,7]))
plt.xticks(rotation=30, ha='right')

legend_lines = [Line2D([0],[0], color=c, linewidth=2, linestyle='--', label=cat)
                for cat, c in cat_colors.items() if cat in ev['category'].values]
ax.legend(handles=legend_lines + [
    Line2D([0],[0], color='#2196F3', linewidth=2.5, marker='o', label='Account Ownership'),
], loc='upper left', fontsize=8, framealpha=0.85, title='Event Category', title_fontsize=9)

ax.set_title('Event Timeline: All Catalogued Events overlaid on Account Ownership Trend',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Adults with Account (%)')
ax.set_facecolor('#F8FBFF')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Event summary table ──────────────────────────────────────────────────────
print('=== All Catalogued Events ===')
ev_display = ev[['record_id','observation_date','category','indicator']].copy()
ev_display['observation_date'] = ev_display['observation_date'].dt.date
display(ev_display.reset_index(drop=True))

---
## 5. Key Insights

Below are **6 key insights** from the EDA, each with supporting evidence and a chart reference.

In [ ]:
insights = [
    {
        'number': 1,
        'title': 'Account ownership growth stalled at its slowest pace ever (2021–2024)',
        'evidence': (
            'Only +3 pp growth (46%→49%) over 2021–2024 — the slowest 3-year window '
            'in the Global Findex series for Ethiopia, despite major product launches.'
        ),
        'implication': (
            'The NFIS-II 70% target by 2025 requires +21 pp in 1 year — almost certainly unachievable. '
            'The trajectory suggests a revised target of ~55–60% by 2026 is more realistic.'
        ),
    },
    {
        'number': 2,
        'title': 'Mobile money registration surged but active usage is low (~30.5%)',
        'evidence': (
            'Mobile money account rate doubled (4.7%→9.45%) driven by Telebirr (launched May 2021) '
            'and M-Pesa (August 2023). Yet NBE data indicates ~30.5% of registered accounts '
            'are active (≥1 transaction/90 days), implying ~6.6M actively used vs. ~21M registered.'
        ),
        'implication': (
            'Policy should focus on usage incentives (merchant acceptance, P2P utility) '
            'not just registration drives.'
        ),
    },
    {
        'number': 3,
        'title': 'P2P transaction volume exploded +158% YoY (FY2023/24→FY2024/25)',
        'evidence': (
            'EthSwitch data: P2P count grew from 49.7M (FY2023/24) to 128.3M (FY2024/25), '
            'and P2P value reached ETB 577.7B. The P2P milestone of surpassing ATM transactions '
            'was crossed in Q1 FY2024/25.'
        ),
        'implication': (
            'Usage is deepening among existing users. The next policy challenge is expanding '
            'the user base to the unregistered 50%, not deepening among the already registered.'
        ),
    },
    {
        'number': 4,
        'title': 'Gender gap is narrowing slowly: 20 pp (2021) → 18 pp (2024)',
        'evidence': (
            'Global Findex data shows male ownership at 56% vs. female at 36% in 2021, '
            'narrowing to an 18 pp gap by 2024. But female MM account share remains only 14% '
            'of total mobile money accounts (NBE/Shega 2024).'
        ),
        'implication': (
            'At current rates, the 50% female MM share target by 2030 (GEN_MM_SHARE target) '
            'is not achievable without targeted gender inclusion programmes.'
        ),
    },
    {
        'number': 5,
        'title': 'Fayda digital ID is the critical access infrastructure unlock',
        'evidence': (
            'Fayda enrollment accelerated from 8M (Aug 2024) to 15M (May 2025) after the '
            'Jan 2024 national rollout. The 90M target by 2028 would cover virtually the entire '
            'adult population, removing the ID barrier for financial service onboarding.'
        ),
        'implication': (
            'Fayda enrollment is a leading indicator for account ownership growth. '
            'Forecasting models should treat Fayda enrollment as a feature for ACCESS indicators.'
        ),
    },
    {
        'number': 6,
        'title': '4G coverage tripled (37%→71%) but data affordability remains a barrier',
        'evidence': (
            'Ethio Telecom data shows 4G population coverage grew from 37.5% (FY2022/23) to '
            '70.8% (FY2024/25). Mobile subscription penetration reached 61.4% (2025). '
            'Yet data affordability index: 2% of GNI/month for 1GB (A4AI threshold is <2%).'
        ),
        'implication': (
            'Infrastructure is no longer the primary bottleneck — affordability and trust are. '
            'FX liberalisation (Jul 2024) increases import costs for devices, compounding affordability.'
        ),
    },
]

print('=' * 72)
print('KEY INSIGHTS — ETHIOPIA FINANCIAL INCLUSION EDA')
print('=' * 72)
for ins in insights:
    print(f"\n📌 Insight {ins['number']}: {ins['title']}")
    print(f"   Evidence   : {ins['evidence']}")
    print(f"   Implication: {ins['implication']}")
print('=' * 72)

---
## 6. Data Limitations and Quality Assessment

In [ ]:
limitations = [
    {
        'category': 'Survey frequency',
        'description': (
            'Global Findex surveys are conducted every 3–4 years (2011, 2014, 2017, 2021, 2024). '
            'This gives only 5 data points for the primary ACCESS indicator over 13 years — '
            'severely limiting time-series forecasting reliability.'
        ),
        'severity': 'High',
    },
    {
        'category': 'Registered vs. active account definition',
        'description': (
            'Findex measures accounts used in the past 12 months; operator data measures '
            'registered subscribers; NBE active-use data uses a 90-day threshold. '
            'These three definitions are not directly comparable.'
        ),
        'severity': 'High',
    },
    {
        'category': 'Missing USAGE and QUALITY indicators',
        'description': (
            'No data on transaction failure rates, system downtime, or merchant acceptance. '
            'QUALITY and TRUST pillars have zero observations in the current dataset.'
        ),
        'severity': 'Medium',
    },
    {
        'category': 'Rural/urban disaggregation',
        'description': (
            'All high-confidence account ownership observations are at the national level. '
            'Ethiopia is 79% rural (World Bank 2023); national averages mask extreme urban-rural gaps.'
        ),
        'severity': 'Medium',
    },
    {
        'category': 'Impact link confidence',
        'description': (
            '8 of 12 enrichment records are impact_links with medium confidence. '
            'These are based on literature/theoretical evidence rather than '
            'Ethiopian empirical data, adding uncertainty to causal inference.'
        ),
        'severity': 'Medium',
    },
    {
        'category': 'Recent data gaps',
        'description': (
            'EthSwitch, NBE, and Ethio Telecom data is available only from FY2023/24 onwards. '
            'There are no USAGE observations before 2024, preventing historical USAGE trend analysis.'
        ),
        'severity': 'Medium',
    },
]

sev_colors = {'High': '🔴', 'Medium': '🟡', 'Low': '🟢'}

print('=== DATA LIMITATIONS AND QUALITY ASSESSMENT ===')
print()
for lim in limitations:
    icon = sev_colors.get(lim['severity'], '⚪')
    print(f"{icon} [{lim['severity']} severity] {lim['category']}")
    print(f"   {lim['description']}")
    print()

# Summary table
summary_df = pd.DataFrame(limitations)[['category', 'severity']]
print('\nSeverity count:')
print(summary_df['severity'].value_counts().to_string())

In [ ]:
# ─── Data quality heatmap: indicators × years ─────────────────────────────────
obs_hm = obs_all.copy()
obs_hm['observation_date'] = pd.to_datetime(obs_hm['observation_date'], errors='coerce')
obs_hm['year'] = obs_hm['observation_date'].dt.year
obs_hm = obs_hm.dropna(subset=['year','indicator_code'])
obs_hm['year'] = obs_hm['year'].astype(int)

pivot = obs_hm.groupby(['indicator_code','year']).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 7))
im = ax.imshow(pivot.values, aspect='auto', cmap='YlGn', vmin=0, vmax=3)

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns.astype(int), fontsize=9, rotation=45, ha='right')
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index, fontsize=9)

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i,j]
        if val > 0:
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=9, fontweight='bold',
                    color='white' if val >= 2 else '#1B5E20')

plt.colorbar(im, ax=ax, shrink=0.6, label='Number of observations')
ax.set_title('Data Coverage Heatmap: Observations per Indicator per Year',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Indicator Code')
plt.tight_layout()
plt.show()

# Identify gaps
all_years = range(int(pivot.columns.min()), int(pivot.columns.max())+1)
total_possible = len(pivot.index) * len(all_years)
total_covered = (pivot > 0).sum().sum()
print(f'\nCoverage rate: {total_covered}/{total_possible} indicator-year combinations = {total_covered/total_possible*100:.0f}%')
print(f'Most sparse indicators (years with 0 observations):')
sparsity = (pivot == 0).sum(axis=1).sort_values(ascending=False)
print(sparsity.head(5).to_string())